In [2]:
import pandas as pd
from pathlib import Path

In [41]:
repo = Path(".").resolve().parent
json_path = repo / "data/raw/json"
print(json_path)

/home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/json


In [42]:
from json_parser_theo import load_chats_from_folder
df = load_chats_from_folder(json_path)
len(df.groupby("chat_id"))

30

In [43]:
df

,chat_id,file_name,turn,role,content
0,1,chat_export,1,user,wie codiere ich alle values in einen pd df zu ...
1,1,chat_export,2,assistant,In pandas kannst du alle negativen Werte im Da...
2,1,chat_export,3,user,Exportiere mir den chat bis hierher als json D...
3,2,copilot,1,user,Was sind wichtige Werke in der Literatur zur r...
4,2,copilot,2,assistant,**Kurzfazit:** \nDie Literatur zur **Reliabil...
...,...,...,...,...,...
284,30,role user content was2,1,user,was ist der unteschied zwischen computer visio...
285,30,role user content was2,2,assistant,"Der zentrale Unterschied liegt darin, **wie Mo..."
286,30,role user content was2,3,user,i need literatur on how to evaluate zero shot ...
287,30,role user content was2,4,assistant,If you're looking specifically for **how to ev...


In [44]:

anzahl_dateien = len([f for f in json_path.iterdir() if f.is_file()])

print(anzahl_dateien)

30


In [46]:
#df.groupby("chat_id")["turn"].max().eq(1)

In [47]:
df_user = df[df["role"] == "user"]
df_user

,chat_id,file_name,turn,role,content
0,1,chat_export,1,user,wie codiere ich alle values in einen pd df zu ...
2,1,chat_export,3,user,Exportiere mir den chat bis hierher als json D...
3,2,copilot,1,user,Was sind wichtige Werke in der Literatur zur r...
6,2,copilot,4,user,Exportiere den gesamten im aktuellen Kontext e...
8,2,copilot,6,user,Exportiere den gesamten sichtbaren Chatverlauf...
...,...,...,...,...,...
281,29,role user content was,1,user,was wären motivationen für die replikation die...
283,29,role user content was,3,user,Gebe mir den gesamten sichtbaren bisherigen Di...
284,30,role user content was2,1,user,was ist der unteschied zwischen computer visio...
286,30,role user content was2,3,user,i need literatur on how to evaluate zero shot ...


In [48]:

df_control = pd.DataFrame({
    "chat_id": range(1, 31),
    "task": ""*30,
    "sentiment": ""*30,
    "critical": ""*30
})

In [ ]:
df_control

In [ ]:
df_control.to_csv(repo / "data/processed/control/control.csv", index=False)
df_user.to_csv(repo / "data/processed/control/user.csv", index=False)

In [32]:
!pip install krippendorff

In [49]:
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import krippendorff

In [87]:
control_a = pd.read_csv(repo / "data/processed/control/control_hannah.csv", delimiter = ";")
control_b = pd.read_csv(repo / "data/processed/control/control_theo_2.csv")
control_c = pd.read_csv(repo / "data/processed/control/control_laura.csv")


In [88]:
def kippendorff_alpha(name: str,
                      annontation = ["code_coder_A", "code_coder_C",  "code_coder_B"],
                        ):
    coder_a = control_a[["chat_id", name]].rename(columns={name: "code_coder_A"})
    coder_b = control_b[["chat_id", name]].rename(columns={name: "code_coder_B"})
    coder_c = control_c[["chat_id", name]].rename(columns={name: "code_coder_C"})
    merged = df_user.merge(coder_a, on="chat_id").merge(coder_b, on="chat_id").merge(coder_c, on="chat_id")
    alpha = krippendorff.alpha(
    reliability_data=merged[annontation].T.to_numpy(),
    level_of_measurement="nominal"
    )
    return alpha


In [89]:
coderABC = ["code_coder_A", "code_coder_C", "code_coder_B"]
coder_AB = ["code_coder_A", "code_coder_B"]
coder_BC = ["code_coder_B", "code_coder_C"]
coder_AC = ["code_coder_A", "code_coder_C"]


for coder in  [coderABC,coder_AB, coder_BC,coder_AC]:
    print("")
    for name in ["task", "sentiment", "critical"]:
        alpha = kippendorff_alpha(name, annontation = coder)
        print(f"Annotators : {coder};" f"Krippendorff's alpha for {name}: {alpha:.4f}")


Annotators : ['code_coder_A', 'code_coder_C', 'code_coder_B'];Krippendorff's alpha for task: 0.6275
Annotators : ['code_coder_A', 'code_coder_C', 'code_coder_B'];Krippendorff's alpha for sentiment: 0.2575
Annotators : ['code_coder_A', 'code_coder_C', 'code_coder_B'];Krippendorff's alpha for critical: 0.3378

Annotators : ['code_coder_A', 'code_coder_B'];Krippendorff's alpha for task: 0.8376
Annotators : ['code_coder_A', 'code_coder_B'];Krippendorff's alpha for sentiment: 0.6626
Annotators : ['code_coder_A', 'code_coder_B'];Krippendorff's alpha for critical: 0.7322

Annotators : ['code_coder_B', 'code_coder_C'];Krippendorff's alpha for task: 0.4494
Annotators : ['code_coder_B', 'code_coder_C'];Krippendorff's alpha for sentiment: 0.1461
Annotators : ['code_coder_B', 'code_coder_C'];Krippendorff's alpha for critical: -0.0135

Annotators : ['code_coder_A', 'code_coder_C'];Krippendorff's alpha for task: 0.6068
Annotators : ['code_coder_A', 'code_coder_C'];Krippendorff's alpha for sentiment

In [77]:
def cohans_kappa(name: str,
                      annontation = ["code_coder_A", "code_coder_C",  "code_coder_B"],
                        ):
    coder_a = control_a[["chat_id", name]].rename(columns={name: "code_coder_A"})
    coder_b = control_b[["chat_id", name]].rename(columns={name: "code_coder_B"})
    coder_b Coder
    coder_c = control_c[["chat_id", name]].rename(columns={name: "code_coder_C"})
    merged = df_user.merge(coder_a, on="chat_id").merge(coder_b, on="chat_id").merge(coder_c, on="chat_id")
    kappa = cohen_kappa_score(merged[annontation[0]], merged[annontation[1]])
    return kappa

In [91]:
print(cohans_kappa(name = "task", annontation = coder_AC))
print(cohans_kappa(name = "sentiment", annontation = coder_AC))
print(cohans_kappa(name = "critical", annontation = coder_AC))

0.6168316831683169
0.012389380530973493
0.20274088725628703
